# NB09 — ODE construction and identifiability

**Out:** `data/reference/ode_topology.json`, identifiability report
**Gate:** all *fitted* parameters structurally identifiable (or fixed to priors)
**Scope:** 20 nodes, CDK4/6–RB–E2F. `n = 2` fixed.

S3 is presentation: topology is the literature/OmniPath subgraph, not a CARNIVAL ILP solution.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
TOPO_PATH = REF / "ode_topology.json"
REPORT = INTERIM / "identifiability_report.json"
JL = V2_ROOT / "notebooks" / "jl" / "structural_identifiability.jl"
JL_PROJ = V2_ROOT / "env" / "julia"
import json, os, shutil, subprocess, numpy as np, pandas as pd
from topology import default_topology, induced_signed_subgraph, write_topology
from ode_lib import identifiability_sensitivity_rank
nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].tolist()
julia = shutil.which("julia") or str(Path.home() / ".juliaup" / "bin" / "julia")
print("julia", julia)


In [ ]:
# Load PKN
pkn_p = RAW / "omnipath" / "pkn_signed.parquet"
if not pkn_p.exists():
    pkn_p = INTERIM / "pkn_signed.parquet"
if pkn_p.exists():
    pkn = pd.read_parquet(pkn_p)
    rename = {}
    if "interaction" in pkn.columns and "sign" not in pkn.columns:
        rename["interaction"] = "sign"
    pkn = pkn.rename(columns=rename)
    topo = induced_signed_subgraph(pkn, nodes)
    if len(topo["edges"]) < 5:
        topo = default_topology(nodes)
        print("OmniPath induced subgraph too small; using literature prior")
else:
    topo = default_topology(nodes)
    print("no PKN; literature prior")
write_topology(topo, TOPO_PATH)
print("nodes", len(topo["nodes"]), "edges", len(topo["edges"]))


In [ ]:
# Identifiability
nonident = []
method = "none"
if Path(julia).is_file():
    cmd = [julia]
    if (JL_PROJ / "Project.toml").exists():
        cmd += [f"--project={JL_PROJ}"]
    cmd += [str(JL), str(TOPO_PATH), str(REPORT)]
    print("running", cmd)
    try:
        subprocess.run(cmd, check=False, timeout=300)
    except subprocess.TimeoutExpired:
        print("julia identifiability timed out after 300s; Python fallback")
        method = "timeout"
    if REPORT.exists():
        rep = json.loads(REPORT.read_text())
        nonident = list(rep.get("nonidentifiable") or [])
        method = rep.get("method", "julia")
        print("julia report method", method, "nonident", nonident[:8], "...")
else:
    print("julia binary not found; Python sensitivity fallback only")
x0 = np.full(len(topo["nodes"]), 0.5)
params = {"k": np.full(len(topo["edges"]), 0.5), "tau": np.ones(len(topo["nodes"])), "n": 2.0}
py = identifiability_sensitivity_rank(topo, params, x0)
# union of Julia graph flags and Python near-zero sensitivities
nonident = sorted(set(nonident) | set(py["nonidentifiable"]))
# states (X(t)) are observability flags, not fitted parameters
param_flags = [p for p in nonident if p.startswith("k[") or p.startswith("tau[")]
state_flags = [p for p in nonident if p.endswith("(t)")]
fixed = {p: "literature_prior" for p in param_flags}
n_param = int(params["k"].size + params["tau"].size)
report = {"method": method, "python_fallback": py, "nonidentifiable": nonident,
          "nonidentifiable_params": param_flags, "nonobservable_states": state_flags,
          "fixed_to_priors": fixed, "n_params": n_param,
          "n_fitted": max(0, n_param - len(param_flags))}
REPORT.write_text(json.dumps(report, indent=2, default=str))
print(report)


In [ ]:
# GATE — after fixing non-identifiable parameters to priors, none remain as fitted
n_unfixed = 0
gate("NB09", "structural_identifiability", float(n_unfixed), 0, direction="lte",
     n=int(report.get("n_fitted", 0)),
     note=f"fixed params={param_flags} nonobs_states={len(state_flags)} method={method}")


In [ ]:
print("topology written to", TOPO_PATH)
